# Model Interpretability — Examples

This notebook demonstrates global and local model interpretation using scikit-learn. It covers permutation importance, partial dependence, and an optional SHAP example.

**Requirements:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`. SHAP is optional.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

RANDOM_STATE = 42


## 1. Fit and evaluate a model
We will explain a model only after checking its performance on a held-out set.

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

model = RandomForestClassifier(
    n_estimators=300, random_state=RANDOM_STATE
)
model.fit(X_train, y_train)
test_prob = model.predict_proba(X_test)[:, 1]
test_pred = model.predict(X_test)

print("Held-out ROC-AUC:", round(roc_auc_score(y_test, test_prob), 4))
print(classification_report(y_test, test_pred, target_names=data.target_names))


## 2. Global explanation: permutation importance
This measures the drop in the chosen score after a feature is shuffled. It is model- and dataset-dependent, and does not establish causality.

In [ ]:
perm = permutation_importance(
    model, X_test, y_test,
    scoring="roc_auc", n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1
)
importance = pd.DataFrame({
    "feature": X.columns,
    "mean_importance": perm.importances_mean,
    "std_importance": perm.importances_std,
}).sort_values("mean_importance", ascending=False)

display(importance.head(10))
top = importance.head(10).sort_values("mean_importance")
plt.barh(top["feature"], top["mean_importance"], xerr=top["std_importance"])
plt.xlabel("Decrease in ROC-AUC after permutation")
plt.title("Top 10 Permutation Importances")
plt.tight_layout()
plt.show()


## 3. Metric dependence
Importance can change when a different evaluation metric is used.

In [ ]:
perm_accuracy = permutation_importance(
    model, X_test, y_test,
    scoring="accuracy", n_repeats=15, random_state=RANDOM_STATE, n_jobs=-1
)
accuracy_importance = pd.Series(
    perm_accuracy.importances_mean, index=X.columns
).sort_values(ascending=False)
display(accuracy_importance.head(10).to_frame("accuracy_importance"))


## 4. Partial dependence
Partial dependence shows an average model response as selected features vary. Correlated features can cause unrealistic feature combinations.

In [ ]:
PartialDependenceDisplay.from_estimator(
    model, X_train, ["mean radius", "mean texture"], kind="average"
)
plt.tight_layout()
plt.show()


## 5. Local prediction
Inspect one test example. The model probability is a model output, not a real-world causal statement.

In [ ]:
row = X_test.iloc[[0]]
probabilities = model.predict_proba(row)[0]
print("Row index:", row.index[0])
print("Predicted class:", model.predict(row)[0], "=>", data.target_names[model.predict(row)[0]])
print("Class probabilities:", dict(zip(data.target_names, probabilities)))
display(row.T.rename(columns={row.index[0]: "feature_value"}))


## 6. Optional SHAP explanation
Install `shap` with `python -m pip install shap` if needed. API output can differ by version; check the explained class/output before interpreting signs.

In [ ]:
try:
    import shap

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(row)
    print("SHAP installed. Returned explanation type:", type(shap_values))
    # For compatible versions, plot a local explanation:
    # shap.force_plot(explainer.expected_value[1], shap_values[1][0], row.iloc[0])
except ImportError:
    print("SHAP is optional and is not installed. Use permutation importance and partial dependence above.")


## 7. Interpretation checklist

- What population does the evaluation set represent?
- Which metric is used for importance?
- Are features strongly correlated?
- Are explanations stable across samples and random seeds?
- Is the output a class, probability, or another score?
- Could leakage or data collection artifacts explain a feature's importance?
- Are conclusions about model behavior being incorrectly stated as causal claims?

## Further reading

- [scikit-learn inspection](https://scikit-learn.org/stable/inspection.html)
- [SHAP documentation](https://shap.readthedocs.io/)
